# Student-t HMM volatility strategy

A Student-t hidden Markov model estimates volatility regimes and maps them to trading positions. The notebook preserves the original training and evaluation design.

Daily OHLCV is downloaded from Yahoo Finance when the notebook runs. Open this notebook from the `notebooks/` directory so it can import the shared data loader.


In [ ]:
"""
================================================================================
CORRECTED STUDENT'S T-DISTRIBUTION HMM TRADING STRATEGY
================================================================================
Key Fixes:
1. Feature standardization (critical!)
2. K-means initialization
3. Diagonal covariance for numerical stability
4. Automatic regime-to-position mapping based on volatility
5. Enhanced error handling
================================================================================
"""

from _market_data import download_ohlcv

import numpy as np
import pandas as pd
from scipy.special import logsumexp, gammaln
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    'TICKER': 'SPY',
    'BENCHMARK': 'URTH',
    'START_DATE': '2019-01-01',
    'END_DATE': '2024-12-31',
    'TRAINING_START': '2020-01-02',
    
    # HMM Configuration - ADJUSTED FOR STABILITY
    'N_REGIMES': 3,
    'MAX_ITER': 100,
    'CONVERGENCE_TOL': 1e-3,
    'RANDOM_SEED': 42,
    'COVARIANCE_TYPE': 'diagonal',  # CRITICAL: Use diagonal for stability
    
    # Feature Configuration
    'SMA_SHORT': 20,
    'SMA_LONG': 200,
    'GK_WINDOW': 20,
    
    # Trading Configuration
    'COMMISSION': 0.001,
    'SLIPPAGE': 0.0005,
    'INITIAL_CAPITAL': 100000,
    
    # AUTO-DETECT regime mapping based on volatility characteristics
    'AUTO_REGIME_MAPPING': True,
    
    # Regularization - CRITICAL FOR NUMERICAL STABILITY
    'COV_REG': 1e-4,
    'MIN_DF': 3.0,
    'MAX_DF': 30.0,
}

# ============================================================================
# IMPROVED STUDENT'S T HMM WITH NUMERICAL STABILITY
# ============================================================================

class RobustStudentTHMM:
    """
    Numerically stable Student's t HMM for financial data
    """
    
    def __init__(self, n_states=3, n_features=5, max_iter=100, tol=1e-3,
                 covariance_type='diagonal', random_state=42, cov_reg=1e-4,
                 min_df=3.0, max_df=30.0):
        self.n_states = n_states
        self.n_features = n_features
        self.max_iter = max_iter
        self.tol = tol
        self.covariance_type = covariance_type
        self.random_state = random_state
        self.cov_reg = cov_reg
        self.min_df = min_df
        self.max_df = max_df
        
        np.random.seed(random_state)
        self.scaler = StandardScaler()
        
        self.pi = None
        self.A = None
        self.mu = None
        self.sigma = None
        self.df = None
        
    def _initialize_kmeans(self, X):
        """Initialize with K-means - much more robust than random"""
        print("  Initializing with K-means clustering...")
        
        kmeans = KMeans(n_clusters=self.n_states, random_state=self.random_state, n_init=10)
        labels = kmeans.fit_predict(X)
        
        # Initial state probabilities
        self.pi = np.ones(self.n_states) / self.n_states
        
        # Transition matrix with persistence (states tend to continue)
        self.A = np.random.rand(self.n_states, self.n_states) * 0.1
        np.fill_diagonal(self.A, 0.7)
        self.A = self.A / self.A.sum(axis=1, keepdims=True)
        
        # Initialize emission parameters from clusters
        self.mu = np.zeros((self.n_states, self.n_features))
        self.sigma = np.zeros((self.n_states, self.n_features))
        
        for k in range(self.n_states):
            mask = labels == k
            if mask.sum() > 1:
                cluster_data = X[mask]
                self.mu[k] = cluster_data.mean(axis=0)
                self.sigma[k] = cluster_data.var(axis=0) + self.cov_reg
            else:
                # Fallback for empty clusters
                self.mu[k] = X.mean(axis=0) + np.random.randn(self.n_features) * 0.1
                self.sigma[k] = X.var(axis=0) + self.cov_reg
        
        # Initialize degrees of freedom (moderate values for finite variance)
        self.df = np.random.uniform(5, 15, size=self.n_states)
        
        print("  ✓ Initialization complete")
    
    def _student_t_logpdf_diagonal(self, X, mu, sigma_diag, df):
        """
        Numerically stable Student's t log-PDF with diagonal covariance
        """
        n, p = X.shape
        
        # Ensure positive variance
        sigma_diag = np.maximum(sigma_diag, 1e-6)
        
        # Mahalanobis distance
        diff = X - mu
        maha_dist = np.sum((diff ** 2) / sigma_diag, axis=1)
        
        # Log determinant
        log_det = np.sum(np.log(sigma_diag))
        
        # Log normalization using gammaln for numerical stability
        log_norm = (
            gammaln((df + p) / 2) -
            gammaln(df / 2) -
            (p / 2) * np.log(df * np.pi) -
            0.5 * log_det
        )
        
        # Log PDF
        log_prob = log_norm - ((df + p) / 2) * np.log(1 + maha_dist / df)
        
        # Handle numerical issues
        log_prob = np.clip(log_prob, -1e10, 1e10)
        log_prob = np.nan_to_num(log_prob, nan=-1e10)
        
        return log_prob
    
    def _emission_probability(self, X):
        """Calculate emission log-probabilities for all states"""
        n_samples = X.shape[0]
        log_prob = np.zeros((n_samples, self.n_states))
        
        for k in range(self.n_states):
            log_prob[:, k] = self._student_t_logpdf_diagonal(
                X, self.mu[k], self.sigma[k], self.df[k]
            )
        
        return log_prob
    
    def _forward(self, log_emission):
        """Forward algorithm in log-space"""
        n_samples = log_emission.shape[0]
        log_alpha = np.zeros((n_samples, self.n_states))
        
        log_alpha[0] = np.log(self.pi + 1e-10) + log_emission[0]
        
        for t in range(1, n_samples):
            for j in range(self.n_states):
                log_alpha[t, j] = logsumexp(
                    log_alpha[t-1] + np.log(self.A[:, j] + 1e-10)
                ) + log_emission[t, j]
        
        return log_alpha
    
    def _backward(self, log_emission):
        """Backward algorithm in log-space"""
        n_samples = log_emission.shape[0]
        log_beta = np.zeros((n_samples, self.n_states))
        
        for t in range(n_samples-2, -1, -1):
            for i in range(self.n_states):
                log_beta[t, i] = logsumexp(
                    np.log(self.A[i, :] + 1e-10) + log_emission[t+1] + log_beta[t+1]
                )
        
        return log_beta
    
    def _compute_gamma_xi(self, log_alpha, log_beta, log_emission):
        """Compute state occupation probabilities"""
        n_samples = log_alpha.shape[0]
        
        # Gamma
        log_gamma = log_alpha + log_beta
        log_gamma = log_gamma - logsumexp(log_gamma, axis=1, keepdims=True)
        gamma = np.exp(log_gamma)
        
        # Handle numerical issues
        gamma = np.nan_to_num(gamma, nan=1/self.n_states)
        gamma = gamma / gamma.sum(axis=1, keepdims=True)
        
        # Xi
        xi = np.zeros((n_samples-1, self.n_states, self.n_states))
        
        for t in range(n_samples-1):
            for i in range(self.n_states):
                for j in range(self.n_states):
                    log_xi = (log_alpha[t, i] + np.log(self.A[i, j] + 1e-10) + 
                             log_emission[t+1, j] + log_beta[t+1, j])
                    xi[t, i, j] = np.exp(log_xi)
            
            xi_sum = xi[t].sum()
            if xi_sum > 1e-10:
                xi[t] = xi[t] / xi_sum
            else:
                xi[t] = 1.0 / (self.n_states ** 2)
        
        return gamma, xi
    
    def fit(self, X, standardize=True):
        """
        Fit HMM with Baum-Welch algorithm
        
        CRITICAL: Always standardize features for numerical stability
        """
        X = np.asarray(X)
        n_samples, n_features = X.shape
        
        # CRITICAL: Standardize features
        if standardize:
            print("  Standardizing features (CRITICAL for numerical stability)...")
            X = self.scaler.fit_transform(X)
        
        # Initialize parameters
        self._initialize_kmeans(X)
        
        print(f"  Training HMM with {n_samples} samples...")
        
        log_likelihood_history = []
        best_log_likelihood = -np.inf
        
        for iteration in range(self.max_iter):
            # E-step
            log_emission = self._emission_probability(X)
            
            # Check for numerical issues
            if np.any(np.isnan(log_emission)) or np.any(np.isinf(log_emission)):
                print(f"  ⚠ Warning: Numerical issues in emission probabilities at iteration {iteration}")
                break
            
            log_alpha = self._forward(log_emission)
            log_beta = self._backward(log_emission)
            
            # Log-likelihood
            log_likelihood = logsumexp(log_alpha[-1])
            
            if np.isnan(log_likelihood) or np.isinf(log_likelihood):
                print(f"  ⚠ Warning: Invalid log-likelihood at iteration {iteration}")
                break
            
            log_likelihood_history.append(log_likelihood)
            
            # Check convergence
            if iteration > 0:
                improvement = log_likelihood - log_likelihood_history[-2]
                
                if iteration % 10 == 0:
                    print(f"    Iteration {iteration}: LL={log_likelihood:.2f}, Improvement={improvement:.4f}")
                
                if abs(improvement) < self.tol:
                    print(f"  ✓ Converged at iteration {iteration+1}, Log-likelihood: {log_likelihood:.2f}")
                    break
            
            # Compute posteriors
            gamma, xi = self._compute_gamma_xi(log_alpha, log_beta, log_emission)
            
            # M-step: Update parameters
            
            # Initial probabilities
            self.pi = gamma[0]
            self.pi = self.pi / self.pi.sum()
            
            # Transition matrix
            gamma_sum = np.maximum(gamma[:-1].sum(axis=0), 1e-10)
            self.A = xi.sum(axis=0) / gamma_sum[:, np.newaxis]
            self.A = self.A / self.A.sum(axis=1, keepdims=True)
            
            # Emission parameters
            for k in range(self.n_states):
                gamma_k = gamma[:, k]
                gamma_k_sum = max(gamma_k.sum(), 1e-10)
                
                # Mean
                self.mu[k] = (gamma_k[:, np.newaxis] * X).sum(axis=0) / gamma_k_sum
                
                # Diagonal variance
                diff = X - self.mu[k]
                self.sigma[k] = (gamma_k[:, np.newaxis] * (diff ** 2)).sum(axis=0) / gamma_k_sum
                self.sigma[k] = np.maximum(self.sigma[k], self.cov_reg)
                
                # Degrees of freedom (keep in reasonable range)
                self.df[k] = np.clip(self.df[k], self.min_df, self.max_df)
            
            best_log_likelihood = max(best_log_likelihood, log_likelihood)
        
        if iteration == self.max_iter - 1:
            print(f"  ⚠ Max iterations reached. Final LL: {log_likelihood:.2f}")
        
        return self
    
    def predict_proba(self, X, standardize=True):
        """Predict state probabilities"""
        X = np.asarray(X)
        
        if standardize:
            X = self.scaler.transform(X)
        
        log_emission = self._emission_probability(X)
        log_alpha = self._forward(log_emission)
        log_beta = self._backward(log_emission)
        
        log_gamma = log_alpha + log_beta
        log_gamma = log_gamma - logsumexp(log_gamma, axis=1, keepdims=True)
        
        gamma = np.exp(log_gamma)
        gamma = np.nan_to_num(gamma, nan=1/self.n_states)
        gamma = gamma / gamma.sum(axis=1, keepdims=True)
        
        return gamma
    
    def predict(self, X, standardize=True):
        """Predict most likely state"""
        proba = self.predict_proba(X, standardize=standardize)
        return np.argmax(proba, axis=1)

# ============================================================================
# FEATURE CALCULATION
# ============================================================================

def calculate_features(df, config):
    """Calculate all HMM features"""
    print("\nCalculating features...")
    df = df.copy()
    
    df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
    df['intraday_returns'] = (df['Close'] / df['Open']) - 1
    df['overnight_returns'] = (df['Open'] / df['Close'].shift(1)) - 1
    
    df['sma_short'] = df['Close'].rolling(window=config['SMA_SHORT']).mean()
    df['sma_long'] = df['Close'].rolling(window=config['SMA_LONG']).mean()
    df['sma_ratio'] = (df['sma_short'] / df['sma_long']) - 1
    
    log_hl = np.log(df['High'] / df['Low'])
    log_co = np.log(df['Close'] / df['Open'])
    gk_variance = 0.5 * (log_hl ** 2) - (2 * np.log(2) - 1) * (log_co ** 2)
    df['gk_volatility'] = np.sqrt(gk_variance.rolling(window=config['GK_WINDOW']).mean() * 252)
    
    feature_cols = ['log_returns', 'intraday_returns', 'overnight_returns', 'sma_ratio', 'gk_volatility']
    df = df.dropna(subset=feature_cols)
    
    print(f"  ✓ {len(df)} data points with complete features")
    return df, feature_cols

# ============================================================================
# AUTOMATIC REGIME MAPPING
# ============================================================================

def determine_regime_positions(hmm_model, X_scaled, config):
    """
    Automatically determine which regimes should be long/short/neutral
    based on regime characteristics (volatility and returns)
    """
    print("\nDetermining optimal regime-to-position mapping...")
    
    # Get regime statistics in original space
    regime_stats = []
    
    for k in range(config['N_REGIMES']):
        # Mean return (feature 0 is log_returns after scaling)
        mean_return = hmm_model.mu[k][0]  # Standardized
        
        # Volatility (feature 4 is gk_volatility)
        volatility = hmm_model.mu[k][4]  # Standardized
        
        regime_stats.append({
            'regime': k,
            'mean_return': mean_return,
            'volatility': volatility,
            'sharpe': mean_return / (abs(volatility) + 1e-6)
        })
    
    # Sort by Sharpe ratio
    regime_stats_sorted = sorted(regime_stats, key=lambda x: x['sharpe'], reverse=True)
    
    # Map: Best Sharpe -> Long, Worst -> Short, Middle -> Neutral
    regime_positions = {}
    
    if config['N_REGIMES'] == 3:
        regime_positions[regime_stats_sorted[0]['regime']] = 1   # Best -> Long
        regime_positions[regime_stats_sorted[1]['regime']] = 0   # Middle -> Neutral
        regime_positions[regime_stats_sorted[2]['regime']] = -1  # Worst -> Short
    else:
        # For other numbers of regimes, map based on Sharpe percentiles
        for i, stats in enumerate(regime_stats_sorted):
            if i < config['N_REGIMES'] // 3:
                regime_positions[stats['regime']] = 1
            elif i > 2 * config['N_REGIMES'] // 3:
                regime_positions[stats['regime']] = -1
            else:
                regime_positions[stats['regime']] = 0
    
    print("\nRegime Mapping:")
    for stats in regime_stats_sorted:
        k = stats['regime']
        pos = regime_positions[k]
        pos_str = {1: "LONG", 0: "NEUTRAL", -1: "SHORT"}[pos]
        print(f"  Regime {k}: {pos_str} (Sharpe: {stats['sharpe']:.3f})")
    
    return regime_positions

# ============================================================================
# BACKTEST ENGINE
# ============================================================================

def run_backtest(df, feature_cols, hmm_model, regime_positions, config):
    """Run backtest with proper T-1 to T flow"""
    print("\nRunning backtest...")
    
    results = df.copy()
    
    # Predict regimes
    X = results[feature_cols].values
    results['regime'] = hmm_model.predict(X, standardize=True)
    results['regime_prob'] = hmm_model.predict_proba(X, standardize=True).max(axis=1)
    
    # Map regimes to positions
    results['signal'] = results['regime'].map(regime_positions)
    
    # Initialize backtest
    results['position'] = 0
    results['portfolio_value'] = config['INITIAL_CAPITAL']
    results['cash'] = config['INITIAL_CAPITAL']
    results['holdings'] = 0.0
    results['trade'] = 0
    results['trade_cost'] = 0.0
    
    position = 0
    holdings = 0.0
    cash = config['INITIAL_CAPITAL']
    
    for i in range(1, len(results)):
        # Signal from T-1
        target_position = results.iloc[i-1]['signal']
        
        # Execute at T open
        open_price = results.iloc[i]['Open']
        close_price = results.iloc[i]['Close']
        
        # Trade if position changes
        if target_position != position:
            portfolio_value_t_minus_1 = results.iloc[i-1]['portfolio_value']
            
            if target_position == 1:  # Go long
                gross_cost = portfolio_value_t_minus_1
                total_cost = gross_cost * (config['COMMISSION'] + config['SLIPPAGE'])
                net_investment = gross_cost - total_cost
                holdings = net_investment / open_price
                cash = 0
                results.iloc[i, results.columns.get_loc('trade')] = 1
                results.iloc[i, results.columns.get_loc('trade_cost')] = total_cost
                
            elif target_position == -1:  # Go short
                gross_value = portfolio_value_t_minus_1
                total_cost = gross_value * (config['COMMISSION'] + config['SLIPPAGE'])
                net_proceeds = gross_value - total_cost
                holdings = -net_proceeds / open_price
                cash = portfolio_value_t_minus_1 + net_proceeds
                results.iloc[i, results.columns.get_loc('trade')] = -1
                results.iloc[i, results.columns.get_loc('trade_cost')] = total_cost
                
            else:  # Go to cash
                if position != 0:
                    gross_proceeds = abs(holdings * open_price)
                    total_cost = gross_proceeds * (config['COMMISSION'] + config['SLIPPAGE'])
                    
                    if position == 1:
                        cash = gross_proceeds - total_cost
                    else:
                        cash = cash - gross_proceeds - total_cost
                    
                    holdings = 0
                    results.iloc[i, results.columns.get_loc('trade')] = -position
                    results.iloc[i, results.columns.get_loc('trade_cost')] = total_cost
            
            position = target_position
        
        # Update portfolio value at T close
        portfolio_value = cash + (holdings * close_price)
        
        results.iloc[i, results.columns.get_loc('position')] = position
        results.iloc[i, results.columns.get_loc('holdings')] = holdings
        results.iloc[i, results.columns.get_loc('cash')] = cash
        results.iloc[i, results.columns.get_loc('portfolio_value')] = portfolio_value
    
    # Calculate returns
    results['strategy_return'] = results['portfolio_value'].pct_change()
    results['buy_hold_return'] = results['Close'].pct_change()
    results['cumulative_strategy'] = (1 + results['strategy_return']).cumprod()
    results['cumulative_buy_hold'] = (1 + results['buy_hold_return']).cumprod()
    
    print(f"  ✓ Backtest complete")
    print(f"  Total trades: {results['trade'].abs().sum()}")
    print(f"  Total costs: ${results['trade_cost'].sum():,.2f}")
    
    return results

# ============================================================================
# PERFORMANCE METRICS
# ============================================================================

def calculate_metrics(results, config):
    """Calculate performance metrics"""
    returns_strategy = results['strategy_return'].dropna()
    returns_buy_hold = results['buy_hold_return'].dropna()
    
    n_years = len(returns_strategy) / 252
    
    metrics = {
        'total_return_strategy': (results['portfolio_value'].iloc[-1] / config['INITIAL_CAPITAL'] - 1) * 100,
        'total_return_buy_hold': (results['Close'].iloc[-1] / results['Close'].iloc[0] - 1) * 100,
    }
    
    metrics['annualized_return_strategy'] = ((1 + metrics['total_return_strategy']/100) ** (1/n_years) - 1) * 100
    metrics['annualized_return_buy_hold'] = ((1 + metrics['total_return_buy_hold']/100) ** (1/n_years) - 1) * 100
    
    metrics['volatility_strategy'] = returns_strategy.std() * np.sqrt(252) * 100
    metrics['volatility_buy_hold'] = returns_buy_hold.std() * np.sqrt(252) * 100
    
    metrics['sharpe_strategy'] = metrics['annualized_return_strategy'] / metrics['volatility_strategy']
    metrics['sharpe_buy_hold'] = metrics['annualized_return_buy_hold'] / metrics['volatility_buy_hold']
    
    # Max drawdown
    cumulative_strategy = results['portfolio_value'] / config['INITIAL_CAPITAL']
    running_max = cumulative_strategy.expanding().max()
    drawdown = (cumulative_strategy - running_max) / running_max
    metrics['max_drawdown_strategy'] = drawdown.min() * 100
    
    cumulative_buy_hold = results['Close'] / results['Close'].iloc[0]
    running_max_bh = cumulative_buy_hold.expanding().max()
    drawdown_bh = (cumulative_buy_hold - running_max_bh) / running_max_bh
    metrics['max_drawdown_buy_hold'] = drawdown_bh.min() * 100
    
    metrics['win_rate'] = (returns_strategy > 0).sum() / len(returns_strategy) * 100
    
    gains = returns_strategy[returns_strategy > 0].sum()
    losses = abs(returns_strategy[returns_strategy < 0].sum())
    metrics['profit_factor'] = gains / losses if losses != 0 else np.inf
    
    metrics['total_trades'] = results['trade'].abs().sum()
    metrics['total_trade_costs'] = results['trade_cost'].sum()
    
    return metrics

def print_report(metrics):
    """Print performance report"""
    print("\n" + "="*80)
    print("PERFORMANCE REPORT")
    print("="*80)
    print(f"\n{'Strategy':^40} | {'Buy & Hold':^37}")
    print("-"*80)
    print(f"Total Return:        {metrics['total_return_strategy']:>8.2f}% | {metrics['total_return_buy_hold']:>8.2f}%")
    print(f"Annualized Return:   {metrics['annualized_return_strategy']:>8.2f}% | {metrics['annualized_return_buy_hold']:>8.2f}%")
    print(f"Volatility:          {metrics['volatility_strategy']:>8.2f}% | {metrics['volatility_buy_hold']:>8.2f}%")
    print(f"Sharpe Ratio:        {metrics['sharpe_strategy']:>8.2f}  | {metrics['sharpe_buy_hold']:>8.2f}")
    print(f"Max Drawdown:        {metrics['max_drawdown_strategy']:>8.2f}% | {metrics['max_drawdown_buy_hold']:>8.2f}%")
    print(f"\nWin Rate:            {metrics['win_rate']:>8.2f}%")
    print(f"Profit Factor:       {metrics['profit_factor']:>8.2f}")
    print(f"Total Trades:        {metrics['total_trades']:>8.0f}")
    print("="*80)

# ============================================================================
# DOWNLOAD DATA FROM FMP
# ============================================================================

def download_price_data(ticker, start, end):
    prices = download_ohlcv(ticker, start, end)
    return prices.rename_axis("date").reset_index()[["date", "Open", "High", "Low", "Close", "Volume"]]

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    print("\n" + "="*80)
    print("MAIN EXECUTION")
    print("="*80)
    
    # Download data
    print("\nDownloading data...")
    spy_data = download_price_data(CONFIG['TICKER'], CONFIG['START_DATE'], CONFIG['END_DATE'])
    
    
    print(f"✓ Downloaded {len(spy_data)} rows")
    
    # Calculate features
    data, feature_cols = calculate_features(spy_data, CONFIG)
    data = data[data['date'] >= CONFIG['TRAINING_START']].reset_index(drop=True)
    
    print(f"\nTraining period: {data['date'].iloc[0].date()} to {data['date'].iloc[-1].date()}")
    print(f"Trading days: {len(data)}")
    
    # Train HMM
    print("\n" + "="*80)
    print("TRAINING HMM")
    print("="*80)
    
    X_train = data.iloc[:500][feature_cols].values
    
    hmm = RobustStudentTHMM(
        n_states=CONFIG['N_REGIMES'],
        n_features=len(feature_cols),
        max_iter=CONFIG['MAX_ITER'],
        tol=CONFIG['CONVERGENCE_TOL'],
        covariance_type=CONFIG['COVARIANCE_TYPE'],
        random_state=CONFIG['RANDOM_SEED'],
        cov_reg=CONFIG['COV_REG'],
        min_df=CONFIG['MIN_DF'],
        max_df=CONFIG['MAX_DF']
    )
    
    hmm.fit(X_train, standardize=True)
    
    # Determine regime mapping
    if CONFIG['AUTO_REGIME_MAPPING']:
        X_scaled = hmm.scaler.transform(data[feature_cols].values)
        regime_positions = determine_regime_positions(hmm, X_scaled, CONFIG)
    else:
        regime_positions = CONFIG['REGIME_POSITIONS']
    
    # Run backtest
    results = run_backtest(data, feature_cols, hmm, regime_positions, CONFIG)
    
    # Calculate metrics
    metrics = calculate_metrics(results, CONFIG)
    
    # Print report
    print_report(metrics)
    
    # Save results
    output_file = f"{CONFIG['TICKER']}_student_t_hmm_corrected.csv"
    results.to_csv(output_file, index=False)
    print(f"\n✓ Results saved to: {output_file}")
    
    return results, metrics, hmm

if __name__ == "__main__":
    results, metrics, hmm_model = main()
